# PyTorch Transformer (Part 2) - Modern Components Practice Workbook

[Open in Colab](https://colab.research.google.com/github/mudit1729/mlwiki-colabs/blob/main/pytorch-code-samples/pytorch-transformer-implementations-2.ipynb)

Practice drills for modern and efficient-inference transformer components: learnable positional encoding, RoPE, LayerNorm/RMSNorm, an end-to-end TinyGPT, KV-cache, grouped-query attention, sliding-window attention, FlashAttention (tiled online softmax), and a Mixture-of-Experts router.

Source wiki page: [transformer implementations part 2](https://auto-driving-wiki.up.railway.app/page/wiki/syntheses/pytorch-code-samples/paper-implementations-transformers-2)

This workbook is **PyTorch-only** and intentionally does **not** include completed answers. Fill the TODOs first, then uncomment each `run_exercise_XX_basic_tests()` call. For the core blocks (attention, MHA, encoder/decoder, ViT, DETR), do the [Transformer Part 1 notebook](https://colab.research.google.com/github/mudit1729/mlwiki-colabs/blob/main/pytorch-code-samples/pytorch-transformer-implementations.ipynb) first.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__)
print("device", device)


def assert_shape(x, shape):
    assert tuple(x.shape) == tuple(shape), f"expected shape {shape}, got {tuple(x.shape)}"


def assert_finite_tensor(x):
    assert torch.isfinite(x).all(), "tensor contains non-finite values"


def assert_close(a, b, atol=1e-5, rtol=1e-5):
    torch.testing.assert_close(a, b, atol=atol, rtol=rtol)


# Workbook setup smoke test.
_x = torch.randn(2, 3, device=device)
assert_shape(_x, (2, 3))
assert_finite_tensor(_x)


## Practice Flow

1. Read the exercise prompt and shape hints.
2. Fill the TODO implementation body.
3. Uncomment `run_exercise_XX_basic_tests()`.
4. Run the cell and fix the implementation until the basic tests pass.

Keep this workbook answer-free while practicing. Add your own edge-case tests after the basic helper passes.


## Exercise 01: Learnable Positional Encoding

Replace fixed sinusoids with a learned position table added to token embeddings.

Expected shapes:

- input `x`: `(batch, seq, d_model)`
- output: same shape as input

Concepts tested: `nn.Embedding` over position indices, broadcasting over the batch, and the `max_len` cap.


In [ ]:
# Exercise 01: Learnable Positional Encoding
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

class LearnablePositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        # TODO: define an nn.Embedding mapping each position index to a d_model vector.

    def forward(self, x):
        """x: (B, T, D); add a learned positional vector to each position."""
        # TODO: build position indices 0..T-1 on x.device.
        # TODO: look up position embeddings and add to x (broadcast over batch).
        raise NotImplementedError("Implement LearnablePositionalEncoding.forward")


def run_exercise_01_tests():
    """Run after filling the TODO implementation above."""
    torch.manual_seed(0)
    max_len, d_model = 32, 16
    pe = LearnablePositionalEncoding(max_len=max_len, d_model=d_model).to(device)

    # Shape / finite / positions actually change the input.
    x = torch.randn(2, 5, d_model, device=device)
    out = pe(x)
    assert_shape(out, (2, 5, d_model))
    assert_finite_tensor(out)
    assert not torch.allclose(out, x)

    # The PE must be LEARNABLE: a parameter table present in module.parameters().
    params = list(pe.parameters())
    assert len(params) >= 1, "positional encoding has no learnable parameters"
    table = next(p for p in params if p.shape == (max_len, d_model))

    # ORACLE: the learned lookup-and-add must equal an nn.Embedding with the SAME
    # (transplanted) weight table, indexed by positions and broadcast over the batch.
    oracle = nn.Embedding(max_len, d_model).to(device)
    with torch.no_grad():
        oracle.weight.copy_(table)
    T = x.shape[1]
    pos = torch.arange(T, device=device)
    expected = x + oracle(pos)            # (1,T,D) broadcast-add over batch
    assert_close(pe(x), expected, atol=1e-6, rtol=1e-6)

    # Broadcast correctness: every batch row gets the identical position offset.
    offset = pe(x) - x
    assert_close(offset[0], offset[1], atol=1e-6, rtol=1e-6)

    # Grad must flow back to the position table.
    pe.zero_grad(set_to_none=True)
    pe(x).pow(2).sum().backward()
    assert table.grad is not None and table.grad.abs().sum() > 0, "no grad to PE table"
    print("exercise 01 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_01_tests()


## Exercise 02: RoPE Rotary Position Embeddings

Implement a RoPE cache and apply rotary embeddings to query/key tensors.

Expected shapes:

- input `x`: `(batch, heads, seq_len, head_dim)`
- `head_dim` must be even
- `cos`, `sin`: broadcastable to `(1, 1, seq_len, head_dim)`
- output: same shape as `x`

Concepts tested: pairwise feature rotation, norm preservation, broadcasting, and LLM-style positional encoding.


In [ ]:
# Exercise 02: RoPE Rotary Position Embeddings
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

def build_rope_cache(seq_len, head_dim, base=10000, device=None, dtype=torch.float32):
    """Return cos, sin caches broadcastable to (B, H, T, Dh)."""
    assert head_dim % 2 == 0, "head_dim must be even for RoPE"
    # TODO: compute inverse frequencies for head_dim//2 pairs.
    # TODO: build angles for each position and frequency.
    # TODO: duplicate/interleave angles so cos/sin match full head_dim.
    # TODO: return cos and sin shaped (1, 1, seq_len, head_dim).
    raise NotImplementedError("Implement build_rope_cache")


def rotate_half(x):
    """Rotate feature pairs: (x0, x1) -> (-x1, x0)."""
    # TODO: split x into even and odd features, rotate pairs, then restore original layout.
    raise NotImplementedError("Implement rotate_half")


def apply_rope(x, cos, sin):
    """Apply RoPE to x using broadcastable cos/sin caches."""
    # TODO: return x * cos + rotate_half(x) * sin.
    raise NotImplementedError("Implement apply_rope")


def run_exercise_02_tests():
    """Run after filling the TODO implementations above."""
    torch.manual_seed(0)
    B, H, T, Dh = 2, 4, 7, 8
    x = torch.randn(B, H, T, Dh, device=device)
    cos, sin = build_rope_cache(seq_len=T, head_dim=Dh, device=device)
    y = apply_rope(x, cos, sin)
    assert_shape(y, x.shape)
    assert_finite_tensor(y)
    assert_shape(cos, (1, 1, T, Dh))
    assert_shape(sin, (1, 1, T, Dh))

    # ORACLE 1 (norm preservation): rotation is orthogonal, so it preserves the
    # per-vector L2 norm at every position.
    assert_close(y.norm(dim=-1), x.norm(dim=-1), atol=1e-5, rtol=1e-5)

    # ORACLE 2 (independent complex-number reference): RoPE on adjacent pairs is a
    # complex multiply by e^{i*m*theta}. Build it independently and compare.
    base = 10000.0
    inv_freq = 1.0 / (base ** (torch.arange(0, Dh, 2, device=device).float() / Dh))  # (Dh/2,)
    pos = torch.arange(T, device=device).float()
    ang = torch.outer(pos, inv_freq)                                # (T, Dh/2)
    xc = torch.view_as_complex(x.float().reshape(B, H, T, Dh // 2, 2))   # pair -> complex
    rot = torch.polar(torch.ones_like(ang), ang)[None, None]            # (1,1,T,Dh/2)
    ref = torch.view_as_real(xc * rot).reshape(B, H, T, Dh)
    assert_close(y, ref, atol=1e-5, rtol=1e-5)

    # ORACLE 3 (relative-position identity): <RoPE(q,m), RoPE(k,n)> depends only on
    # (m-n). Equal offsets -> equal dot products; different offset -> different.
    torch.manual_seed(1)
    q = torch.randn(1, 1, 1, Dh, device=device)
    k = torch.randn(1, 1, 1, Dh, device=device)

    def rope_at(vec, m):
        c, s = build_rope_cache(seq_len=m + 1, head_dim=Dh, device=device)
        return apply_rope(vec, c[:, :, m:m + 1], s[:, :, m:m + 1])

    def dot(m, n):
        return (rope_at(q, m) * rope_at(k, n)).sum()

    # offset +1 at two different absolute positions must match.
    assert_close(dot(3, 2), dot(5, 4), atol=1e-5, rtol=1e-5)
    # offset -2 at two different absolute positions must match.
    assert_close(dot(2, 4), dot(4, 6), atol=1e-5, rtol=1e-5)
    # a different offset must give a different dot product.
    assert not torch.allclose(dot(3, 2), dot(5, 2), atol=1e-4, rtol=1e-4)

    # Grad flows through the rotation.
    xg = x.clone().requires_grad_(True)
    apply_rope(xg, cos, sin).pow(2).sum().backward()
    assert xg.grad is not None and torch.isfinite(xg.grad).all()
    print("exercise 02 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_02_tests()


## Exercise 03: Layer Normalization And RMSNorm

Implement LayerNorm (the transformer variant) and RMSNorm from scratch.

Expected shapes: output matches input `(batch, seq, d_model)`.

Concepts tested: per-token normalization over the feature dim, learned scale/shift, biased variance (to match `nn.LayerNorm`), and RMSNorm dropping mean-centering and bias.


In [ ]:
# Exercise 03: Layer Normalization And RMSNorm
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        # TODO: define learned scale (gamma, ones) and shift (beta, zeros).

    def forward(self, x):
        # TODO: normalize over the last dim using mean and biased variance, then scale/shift.
        raise NotImplementedError("Implement LayerNorm.forward")


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        # TODO: define learned scale (gamma, ones); RMSNorm has no shift.

    def forward(self, x):
        # TODO: divide x by the root-mean-square over the last dim (no mean-centering), then scale.
        raise NotImplementedError("Implement RMSNorm.forward")


def run_exercise_03_tests():
    """Run after filling the TODO implementations above."""
    torch.manual_seed(0)
    d_model = 16
    x = torch.randn(4, 7, d_model, device=device)

    # ORACLE (LayerNorm): transplant gamma/beta into nn.LayerNorm and match tightly,
    # including a NON-identity affine so scale+shift are both exercised.
    ln = LayerNorm(d_model).to(device)
    with torch.no_grad():
        ln.gamma.copy_(torch.randn(d_model, device=device))
        ln.beta.copy_(torch.randn(d_model, device=device))
    ref_ln = nn.LayerNorm(d_model).to(device)
    with torch.no_grad():
        ref_ln.weight.copy_(ln.gamma)
        ref_ln.bias.copy_(ln.beta)
    assert_close(ln(x), ref_ln(x), atol=1e-5, rtol=1e-5)

    # No-affine case: identity gamma=1, beta=0 must match nn.LayerNorm defaults.
    ln_id = LayerNorm(d_model).to(device)
    assert_close(ln_id(x), nn.LayerNorm(d_model).to(device)(x), atol=1e-5, rtol=1e-5)

    # ORACLE (RMSNorm): transplant gamma into nn.RMSNorm and F.rms_norm.
    rms = RMSNorm(d_model, eps=1e-6).to(device)
    with torch.no_grad():
        rms.gamma.copy_(torch.randn(d_model, device=device))
    ref_rms = nn.RMSNorm(d_model, eps=1e-6).to(device)
    with torch.no_grad():
        ref_rms.weight.copy_(rms.gamma)
    assert_close(rms(x), ref_rms(x), atol=1e-5, rtol=1e-5)
    assert_close(rms(x), F.rms_norm(x, (d_model,), rms.gamma, eps=1e-6), atol=1e-5, rtol=1e-5)

    # No-weight RMSNorm (gamma=1) matches F.rms_norm with no weight.
    rms_id = RMSNorm(d_model, eps=1e-6).to(device)
    assert_close(rms_id(x), F.rms_norm(x, (d_model,), None, eps=1e-6), atol=1e-5, rtol=1e-5)

    # Output properties + grad flow.
    y = rms(x)
    assert_shape(y, (4, 7, d_model))
    assert_finite_tensor(y)
    ln.zero_grad(set_to_none=True)
    ln(x).pow(2).sum().backward()
    assert ln.gamma.grad is not None and ln.beta.grad is not None
    print("exercise 03 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_03_tests()


## Exercise 04: Tiny Transformer End-to-End (TinyGPT)

Assemble a complete decoder-only language model: token + learned position embeddings, pre-norm causal blocks, a final norm, and a tied LM head.

Expected shapes: `forward(idx)` with `idx` of shape `(batch, seq)` returns logits `(batch, seq, vocab_size)`.

Concepts tested: causal self-attention, pre-norm residual blocks, weight tying (the LM head reuses the embedding matrix), and an end-to-end next-token loss.


In [ ]:
# Exercise 04: Tiny Transformer End-to-End (TinyGPT)
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)   # fused Q,K,V
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        # TODO: project to q,k,v and split into (B, n_heads, T, head_dim).
        # TODO: scaled dot-product scores; apply a causal mask (no peeking ahead).
        # TODO: softmax, weighted sum over v, merge heads, output projection.
        raise NotImplementedError("Implement CausalSelfAttention.forward")


class Block(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=4):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model), nn.GELU(),
            nn.Linear(mlp_ratio * d_model, d_model),
        )

    def forward(self, x):
        # TODO: pre-norm residual attention, then pre-norm residual MLP.
        raise NotImplementedError("Implement Block.forward")


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, max_len=128):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        # TODO: tie weights -- make the LM head reuse the token embedding matrix.

    def forward(self, idx):
        """idx: (B, T) token ids -> logits (B, T, vocab_size)."""
        # TODO: add token and position embeddings.
        # TODO: run the blocks, final norm, and LM head.
        raise NotImplementedError("Implement TinyGPT.forward")


def run_exercise_04_tests():
    """Run after filling the TODO implementations above."""
    torch.manual_seed(0)
    vocab = 50
    model = TinyGPT(vocab_size=vocab).to(device)
    idx = torch.randint(0, vocab, (2, 16), device=device)

    # Shape + finite.
    logits = model(idx)
    assert_shape(logits, (2, 16, vocab))
    assert_finite_tensor(logits)

    # Weight tying: LM head IS the embedding matrix.
    assert model.head.weight is model.tok_emb.weight

    # CAUSALITY ORACLE: perturbing a future token must not change earlier-position
    # logits (decoder-only models cannot peek ahead).
    model.eval()
    with torch.no_grad():
        base = model(idx)
        idx2 = idx.clone()
        t_pert = 10
        idx2[:, t_pert] = (idx2[:, t_pert] + 1) % vocab          # change token at t_pert
        pert = model(idx2)
    assert_close(base[:, :t_pert], pert[:, :t_pert], atol=1e-5, rtol=1e-5)  # earlier unchanged
    assert not torch.allclose(base[:, t_pert], pert[:, t_pert])             # this/later changed

    # GRAD FLOW: every parameter receives a gradient end-to-end.
    model.train()
    model.zero_grad(set_to_none=True)
    loss = F.cross_entropy(model(idx[:, :-1]).reshape(-1, vocab), idx[:, 1:].reshape(-1))
    loss.backward()
    for name, p in model.named_parameters():
        assert p.grad is not None, f"no grad for {name}"

    # TRAINING SMOKE TEST: loss must decrease on a fixed batch over a few SGD steps.
    torch.manual_seed(0)
    small = TinyGPT(vocab_size=vocab).to(device)
    small.train()
    opt = torch.optim.Adam(small.parameters(), lr=1e-2)
    batch = torch.randint(0, vocab, (4, 12), device=device)
    losses = []
    for _ in range(25):
        opt.zero_grad()
        lg = small(batch[:, :-1])
        l = F.cross_entropy(lg.reshape(-1, vocab), batch[:, 1:].reshape(-1))
        l.backward()
        opt.step()
        losses.append(l.item())
    assert losses[-1] < losses[0], f"loss did not decrease: {losses[0]:.3f} -> {losses[-1]:.3f}"
    print("exercise 04 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_04_tests()


## Exercise 05: Key-Value Cache (KV-Cache) For Inference

Implement a KV-cache and a cached attention step so autoregressive decoding reuses past keys/values instead of recomputing them.

Expected behavior: feeding tokens one at a time with a cache must match a full causal forward pass over the whole sequence.

Concepts tested: appending K/V along the time dim, why a single in-flight token needs no causal mask, and `O(T)` vs `O(T^2)` decode cost.


In [ ]:
# Exercise 05: Key-Value Cache (KV-Cache) For Inference
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

class KVCache:
    def __init__(self):
        self.k, self.v = None, None

    def update(self, k, v):
        """Append new k,v (B,H,T_new,Dh) to the cache and return the full k,v."""
        # TODO: concatenate along the time dimension (dim=2), handling the empty cache.
        raise NotImplementedError("Implement KVCache.update")


class CachedAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, cache=None):
        """x: (B, T_new, D); during incremental decode T_new == 1."""
        # TODO: project + split heads into (B, n_heads, T_new, head_dim).
        # TODO: if cache is given, update it so k,v span the full prefix.
        # TODO: attention of the new queries over all keys (one token needs no mask), merge, project.
        raise NotImplementedError("Implement CachedAttention.forward")


def run_exercise_05_tests():
    """Run after filling the TODO implementations above."""
    torch.manual_seed(1)
    attn = CachedAttention(32, 4).to(device).eval()

    # DIFFERENTIAL ORACLE: token-by-token cached decode must equal a single full
    # causal forward over the whole sequence (an independent reference built from
    # attn's own weights). Cover a couple of sequence lengths.
    def reference_causal_attention(attn, seq):
        B, T, D = seq.shape
        H, Dh = attn.n_heads, attn.head_dim
        q, k, v = (t.reshape(B, T, H, Dh).transpose(1, 2) for t in attn.qkv(seq).chunk(3, -1))
        s = q @ k.transpose(-2, -1) / math.sqrt(Dh)
        s = s.masked_fill(~torch.tril(torch.ones(T, T, dtype=torch.bool, device=seq.device)),
                          float("-inf"))
        return attn.proj((s.softmax(-1) @ v).transpose(1, 2).reshape(B, T, D))

    for T in (1, 5, 9):
        seq = torch.randn(2, T, 32, device=device)
        with torch.no_grad():
            cache = KVCache()
            incremental = torch.cat([attn(seq[:, t:t + 1], cache=cache) for t in range(T)], dim=1)
            full = reference_causal_attention(attn, seq)
        assert_shape(incremental, (2, T, 32))
        assert_close(incremental, full, atol=1e-5, rtol=1e-5)  # cached decode == full causal pass

    # The cache must actually accumulate K/V along the time dim.
    cache = KVCache()
    with torch.no_grad():
        for t in range(4):
            attn(torch.randn(2, 1, 32, device=device), cache=cache)
    assert cache.k.shape[2] == 4 and cache.v.shape[2] == 4, "cache did not grow along time"
    print("exercise 05 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_05_tests()


## Exercise 06: Grouped Query Attention (GQA)

Implement attention with `n_heads` query heads but only `n_kv_heads` key/value heads (each shared by a group of query heads).

Expected shape: `(batch, seq, d_model)`.

Concepts tested: separate Q vs K/V projections, expanding KV heads across their query group (`repeat_interleave`), causal masking, and the KV-cache savings of GQA/MQA.


In [ ]:
# Exercise 06: Grouped Query Attention (GQA)
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.group = n_heads // n_kv_heads                       # query heads per KV head
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.kv_proj = nn.Linear(d_model, 2 * n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, is_causal=True):
        B, T, _ = x.shape
        # TODO: project queries -> (B, n_heads, T, head_dim).
        # TODO: project keys/values -> (B, n_kv_heads, T, head_dim) each.
        # TODO: expand each KV head to its query group (repeat_interleave by self.group).
        # TODO: scaled scores, optional causal mask, softmax, weighted sum, merge, out_proj.
        raise NotImplementedError("Implement GroupedQueryAttention.forward")


def run_exercise_06_tests():
    """Run after filling the TODO implementation above."""
    torch.manual_seed(0)
    d_model, n_heads = 32, 4
    x = torch.randn(2, 6, d_model, device=device)

    # Shape / finite.
    gqa = GroupedQueryAttention(d_model, n_heads, n_kv_heads=2).to(device)
    out = gqa(x)
    assert_shape(out, (2, 6, d_model))
    assert_finite_tensor(out)

    # DIFFERENTIAL ORACLE (reduces to MHA): with n_kv_heads == n_heads there is no
    # KV sharing, so GQA must equal standard multi-head attention. Build an MHA
    # reference via F.scaled_dot_product_attention from the SAME projections.
    def mha_reference(mod, x, is_causal=True):
        B, T, _ = x.shape
        H, Dh, G = mod.n_heads, mod.head_dim, mod.n_kv_heads
        q = mod.q_proj(x).reshape(B, T, H, Dh).transpose(1, 2)
        kv = mod.kv_proj(x).reshape(B, T, 2 * G, Dh).transpose(1, 2)
        k, v = kv.chunk(2, dim=1)
        k = k.repeat_interleave(H // G, dim=1)
        v = v.repeat_interleave(H // G, dim=1)
        o = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)
        return mod.out_proj(o.transpose(1, 2).reshape(B, T, -1))

    full_kv = GroupedQueryAttention(d_model, n_heads, n_kv_heads=n_heads).to(device)
    assert_close(full_kv(x), mha_reference(full_kv, x), atol=1e-5, rtol=1e-5)

    # n_kv_heads == 1 is Multi-Query Attention -> still equals the SDPA reference.
    mqa = GroupedQueryAttention(d_model, n_heads, n_kv_heads=1).to(device)
    assert_close(mqa(x), mha_reference(mqa, x), atol=1e-5, rtol=1e-5)

    # The GQA path itself must match an explicit SDPA reference for the 2-KV case.
    assert_close(gqa(x), mha_reference(gqa, x), atol=1e-5, rtol=1e-5)

    # Non-causal path also matches.
    assert_close(gqa(x, is_causal=False), mha_reference(gqa, x, is_causal=False),
                 atol=1e-5, rtol=1e-5)

    # Grad flow to every parameter.
    gqa.zero_grad(set_to_none=True)
    gqa(x).pow(2).sum().backward()
    for name, p in gqa.named_parameters():
        assert p.grad is not None, f"no grad for {name}"
    print("exercise 06 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_06_tests()


## Exercise 07: Sliding Window Attention

Implement local causal attention where each token attends only to the previous `window` keys.

Expected shapes: mask `(seq, seq)` boolean; attention output `(batch, seq, d_model)`.

Concepts tested: combining causal and local constraints into one mask, and `O(T*W)` vs `O(T^2)` cost.


In [ ]:
# Exercise 07: Sliding Window Attention
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

def sliding_window_mask(T, window, device=None):
    """(T, T) bool keep-mask: token i attends to keys j with 0 <= i - j < window."""
    # TODO: build a (T, T) mask that is causal AND within the last `window` keys.
    raise NotImplementedError("Implement sliding_window_mask")


class SlidingWindowAttention(nn.Module):
    def __init__(self, d_model, n_heads, window):
        super().__init__()
        self.n_heads, self.head_dim, self.window = n_heads, d_model // n_heads, window
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        # TODO: project + split heads, compute scaled scores.
        # TODO: apply sliding_window_mask before softmax, then weighted sum, merge, project.
        raise NotImplementedError("Implement SlidingWindowAttention.forward")


def run_exercise_07_tests():
    """Run after filling the TODO implementations above."""
    # Mask structure for a small case.
    mask = sliding_window_mask(6, 3, device=device)
    assert_shape(mask, (6, 6))
    assert mask.dtype == torch.bool
    assert mask[5, 5].item() and mask[5, 3].item()    # inside the window
    assert not mask[5, 2].item()                       # outside the window
    assert not mask[0, 1].item()                       # no future keys

    # ORACLE (explicit band mask): the keep-mask must equal an independently built
    # banded causal mask for several windows, including the boundary window=1.
    for T, window in [(7, 1), (7, 2), (8, 4)]:
        m = sliding_window_mask(T, window, device=device)
        i = torch.arange(T, device=device)[:, None]
        j = torch.arange(T, device=device)[None, :]
        expected = (i >= j) & (i - j < window)
        assert torch.equal(m, expected), f"band mask wrong for T={T}, window={window}"
    # window=1 means each token attends to itself only -> identity mask.
    assert torch.equal(sliding_window_mask(5, 1, device=device),
                       torch.eye(5, dtype=torch.bool, device=device))

    torch.manual_seed(0)
    d_model, n_heads = 32, 4
    x = torch.randn(2, 6, d_model, device=device)
    out = SlidingWindowAttention(d_model, n_heads, window=3).to(device)(x)
    assert_shape(out, (2, 6, d_model))
    assert_finite_tensor(out)

    # DIFFERENTIAL ORACLE (window >= T == full causal): with a window covering the
    # whole sequence, sliding-window attention must equal full causal attention
    # (F.scaled_dot_product_attention with is_causal=True) on the same weights.
    torch.manual_seed(3)
    swa_full = SlidingWindowAttention(d_model, n_heads, window=64).to(device)
    xs = torch.randn(2, 6, d_model, device=device)
    B, T, D = xs.shape
    H, Dh = swa_full.n_heads, swa_full.head_dim
    q, k, v = (t.reshape(B, T, H, Dh).transpose(1, 2) for t in swa_full.qkv(xs).chunk(3, -1))
    ref = swa_full.proj(F.scaled_dot_product_attention(q, k, v, is_causal=True)
                        .transpose(1, 2).reshape(B, T, D))
    assert_close(swa_full(xs), ref, atol=1e-5, rtol=1e-5)

    # Grad flow to every parameter.
    swa = SlidingWindowAttention(d_model, n_heads, window=3).to(device)
    swa.zero_grad(set_to_none=True)
    swa(x).pow(2).sum().backward()
    for name, p in swa.named_parameters():
        assert p.grad is not None, f"no grad for {name}"
    print("exercise 07 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_07_tests()


## Exercise 08: Flash Attention (Simplified Tiling)

Implement memory-light attention that streams over key/value blocks with an online softmax (running max + denominator), never materializing the full `T x T` score matrix.

Expected shapes: `q, k, v`: `(B, H, T, Dh)`; output: `(B, H, T, Dh)`.

Concepts tested: the online-softmax recurrence (rescale running accumulators when the max grows) and equivalence to standard softmax attention.


In [ ]:
# Exercise 08: Flash Attention (Simplified Tiling)
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

def flash_attention(q, k, v, block_size=16):
    """Memory-light attention via online softmax over key/value blocks. q,k,v: (B,H,T,Dh)."""
    B, H, Tq, Dh = q.shape
    scale = 1.0 / math.sqrt(Dh)
    out = torch.zeros_like(q)
    m = torch.full((B, H, Tq, 1), float("-inf"), device=q.device)  # running row max
    l = torch.zeros((B, H, Tq, 1), device=q.device)               # running softmax denominator
    for start in range(0, k.shape[2], block_size):
        kb, vb = k[:, :, start:start + block_size], v[:, :, start:start + block_size]
        # TODO: block scores s = (q @ kb^T) * scale.
        # TODO: m_new = max(m, rowmax(s)); p = exp(s - m_new); correction = exp(m - m_new).
        # TODO: l = correction * l + sum(p); out = correction * out + p @ vb; m = m_new.
        raise NotImplementedError("Implement the flash_attention online-softmax update")
    return out / l


def run_exercise_08_tests():
    """Run after filling the TODO implementation above."""
    torch.manual_seed(2)
    B, H, T, Dh = 2, 3, 40, 8
    q, k, v = (torch.randn(B, H, T, Dh, device=device) for _ in range(3))

    out = flash_attention(q, k, v, block_size=16)
    assert_shape(out, (B, H, T, Dh))
    assert_finite_tensor(out)

    # DIFFERENTIAL ORACLE (the whole point of flash): tiled online-softmax output
    # must EQUAL exact attention. Check against BOTH a naive softmax reference and
    # F.scaled_dot_product_attention, to tight tolerance.
    naive = (q @ k.transpose(-2, -1) / math.sqrt(Dh)).softmax(-1) @ v
    assert_close(out, naive, atol=1e-5, rtol=1e-5)  # must equal standard softmax attention
    assert_close(out, F.scaled_dot_product_attention(q, k, v), atol=1e-5, rtol=1e-5)

    # Equality must hold across block sizes (including a block that doesn't divide T,
    # and a single full block) -- online softmax is invariant to the tiling.
    for bs in (1, 7, 16, 64):
        assert_close(flash_attention(q, k, v, block_size=bs),
                     F.scaled_dot_product_attention(q, k, v), atol=1e-5, rtol=1e-5)

    # GRAD FLOW: gradients must match autograd through exact attention.
    qg = q.clone().requires_grad_(True)
    flash_attention(qg, k, v, block_size=16).pow(2).sum().backward()
    qg2 = q.clone().requires_grad_(True)
    F.scaled_dot_product_attention(qg2, k, v).pow(2).sum().backward()
    assert qg.grad is not None and torch.isfinite(qg.grad).all()
    assert_close(qg.grad, qg2.grad, atol=1e-4, rtol=1e-4)
    print("exercise 08 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_08_tests()


## Exercise 09: Mixture Of Experts (MoE) Router

Implement a router that sends each token to its top-k experts and combines their outputs weighted by the (renormalized) gate probabilities.

Expected shape: `(batch, seq, d_model)` (unchanged).

Concepts tested: top-k gating, renormalizing the chosen weights, running only k of E experts per token, and keeping the op differentiable.


In [ ]:
# Exercise 09: Mixture Of Experts (MoE) Router
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

class MoERouter(nn.Module):
    def __init__(self, d_model, n_experts, d_ff, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.gate = nn.Linear(d_model, n_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
            for _ in range(n_experts)
        ])

    def forward(self, x):
        # TODO: gate logits -> softmax -> top_k weights and expert indices.
        # TODO: renormalize the top_k weights so they sum to 1 per token.
        # TODO: for each selected slot/expert, add weight * expert(tokens routed there).
        raise NotImplementedError("Implement MoERouter.forward")


def run_exercise_09_tests():
    """Run after filling the TODO implementation above."""
    torch.manual_seed(0)
    d_model, n_experts, top_k = 16, 4, 2
    moe = MoERouter(d_model=d_model, n_experts=n_experts, d_ff=32, top_k=top_k).to(device)
    x = torch.randn(2, 5, d_model, device=device, requires_grad=True)

    out = moe(x)
    assert_shape(out, (2, 5, d_model))
    assert_finite_tensor(out)

    # ORACLE (selected experts): the router's chosen expert indices must match
    # torch.topk over the gate logits (softmax is monotonic, so topk agrees).
    gate_logits = moe.gate(x)
    probs = gate_logits.softmax(-1)
    top_w, top_i = torch.topk(probs, top_k, dim=-1)
    ref_idx = torch.topk(gate_logits, top_k, dim=-1).indices
    assert torch.equal(top_i, ref_idx), "selected experts disagree with torch.topk"

    # Combine weights over the top-k must be a renormalized softmax summing to 1.
    combine = top_w / top_w.sum(-1, keepdim=True)
    assert_close(combine.sum(-1), torch.ones(2, 5, device=device), atol=1e-6, rtol=1e-6)

    # ORACLE (output value): rebuild the expected output from the renormalized
    # top-k weights and the corresponding experts; the router output must match.
    expected = torch.zeros_like(x)
    for slot in range(top_k):
        for e in range(n_experts):
            tok_mask = top_i[..., slot] == e
            if tok_mask.any():
                expected[tok_mask] += combine[..., slot][tok_mask, None] * moe.experts[e](x[tok_mask])
    assert_close(out, expected, atol=1e-5, rtol=1e-5)

    # Load-balancing aux loss (Switch-style): fraction of tokens per expert times
    # mean router prob per expert, scaled by n_experts -- finite and non-negative.
    one_hot = F.one_hot(top_i[..., 0], n_experts).float()
    load = one_hot.mean(dim=(0, 1))                  # fraction routed to each expert
    importance = probs.mean(dim=(0, 1))              # mean gate prob per expert
    aux_loss = n_experts * (load * importance).sum()
    assert torch.isfinite(aux_loss) and aux_loss.item() >= 0.0

    # GRAD FLOW: through the gate (router stays differentiable) and back to the input.
    moe.zero_grad(set_to_none=True)
    out.sum().backward()
    assert x.grad is not None  # router must stay differentiable
    assert moe.gate.weight.grad is not None and moe.gate.weight.grad.abs().sum() > 0
    print("exercise 09 tests passed")

# Uncomment after implementing the exercise:
# run_exercise_09_tests()


## After You Finish

Compare your implementations against the reference page:

- [[wiki/syntheses/pytorch-code-samples/paper-implementations-transformers-2]]
- [[wiki/syntheses/pytorch-code-samples/paper-implementations-transformers]] (Part 1: core blocks)

Follow-up drills:

1. Combine RoPE (Ex02) with the TinyGPT attention (Ex04) by rotating Q/K before scores.
2. Add a KV-cache to TinyGPT and generate text autoregressively.
3. Swap LayerNorm for RMSNorm inside the TinyGPT blocks.
4. Give GQA a KV-cache and measure the memory saved vs full multi-head attention.
5. Add a load-balancing auxiliary loss to the MoE router.
